# Ambiguity

In [1]:
import nltk
from nltk import Tree

## Custom Tokenizer and parse tree builder

In [2]:
def tokenize(sentence) -> list:
    """
    tokenize a sentence
    :param sentence: the sentence
    :return: a list of tokens
    """
    return sentence.split()

In [3]:
def build_parse_tree(tokens) -> Tree:
    """
    Simple syntax tree manually defined
    :param tokens: the tokenized sentence
    :return:
    """
    if not tokens:
        return None
    if len(tokens) == 1:
        return Tree(tokens[0], [])
    root_index = len(tokens)//2
    root = tokens[root_index]
    left_subtree = build_parse_tree(tokens[:root_index])
    right_subtree = build_parse_tree(tokens[root_index+1:])
    children = [child for child in [left_subtree, right_subtree] if child is not None]
    return Tree(root, children)

In [4]:
tokens = tokenize("The quick brown fox jumps over the lazy dog")
tree = build_parse_tree(tokens)
tree.pretty_print()

                jumps              
         _________|_________        
      brown                lazy    
   _____|____           ____|____   
quick        |        the        | 
  |          |         |         |  
 The        fox       over      dog
  |          |         |         |  
 ...        ...       ...       ...




#### We can recognize Parts of Speech (syntax) using a simple dictionary-based approach, and save the results in a DataFrame.

This can prove useful in order to analyze the structure of a sentence and address ambiguity, in order to then train models to predict the correct POS tags.

In [5]:
if len(tokens) >= 9:
    roles = {
        tokens[0]:"det", tokens[1]:"adj", tokens[2]:"adj",tokens[3]:"nsub", tokens[4]:"verb", tokens[5]:"prep",
        tokens[6]:"det", tokens[7]:"adj", tokens[8]:"obj"
    }
else:
    roles = {token: "unknown" for token in tokens}

for word, role in roles.items():
    print(f"{word} -> {role}")

The -> det
quick -> adj
brown -> adj
fox -> nsub
jumps -> verb
over -> prep
the -> det
lazy -> adj
dog -> obj


In [8]:
word_list = tokens
word_dict = {i: word for i, word in enumerate(tokens)}
print("Array (index based representation)")

for key, value in word_dict.items():
    print(f"{key} -> {value}")

Array (index based representation)
0 -> The
1 -> quick
2 -> brown
3 -> fox
4 -> jumps
5 -> over
6 -> the
7 -> lazy
8 -> dog


In [11]:
import pandas as pd

df = pd.DataFrame( {
    'Index': list(word_dict.keys()),
    'Word': list(word_dict.values()),
    'Role': [roles.get(word, "unknown") for word in word_dict.values()]
})

df.describe()
df.head()


,Index,Word,Role
0,0,The,det
1,1,quick,adj
2,2,brown,adj
3,3,fox,nsub
4,4,jumps,verb


##### POS parsing can be achieved using Stanford's NLP library, which provides a simple interface to tokenize and tag POS.

In [12]:
import stanza
nlp = stanza.Pipeline(lang='en', processors='tokenize, pos') # pos = Part of Speech



C:\Users\claza\anaconda3\envs\NLP2025\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\claza\anaconda3\envs\NLP2025\Lib\site-packages\stanza\models\constituency\parse_transitions.py:322: SyntaxWarning: invalid escape sequence '\T'
  return "\Tree [.%s ? ]" % self.label
2025-03-24 15:42:18 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2025-03-24 15:42:18 INFO: Downloaded file to C:\Users\claza\stanza_resources\resources.json
2025-03-24 15:42:18 WARNING: Language en package default expects mwt, which has been added
2025-03-24 15:42:19 INFO: Loading these models for language: en (English):
| Processor | Package         |
-------------------------------
| tokenize 

In [19]:
def stanza_pos_parser(s) -> tuple:
    """
    Tokenize and POS Tagging using Stanza
    :param s:  the sentence to be analyzed
    :return: a tuple with tokens and POS tags
    """
    doc = nlp(s)
    tokens = [w.text for sent in doc.sentences for w in sent.words]
    pos_tags = [(w.text, w.pos) for sent in doc.sentences for w in sent.words]

    print("Stanza pipeline tokenization and POS Tagging")
    #for token, pos in pos_tags:
    #    print(f"{token} -> {pos}")
    return tokens, pos_tags

In [22]:
sentence = "The quick brown fox jumps over the lazy dog"
tokens, pos_tags = stanza_pos_parser(sentence)

Stanza pipeline tokenization and POS Tagging


In [25]:
stanza_dataframe = pd.DataFrame({
    'Token': [token for token, pos in pos_tags],
    'POS': [pos for token, pos in pos_tags]
})

stanza_dataframe_better = pd.DataFrame(pos_tags, columns=['Token', 'POS'])
stanza_dataframe_better.head()

,Token,POS
0,The,DET
1,quick,ADJ
2,brown,ADJ
3,fox,NOUN
4,jumps,VERB
